# Methodological Review & Leakage Analysis

## Detected Issues
1. **Target Data Leakage (Data Snooping)**: The `job_yes_rate` feature (Target Encoding) is calculated using the entire dataset (`working_df.groupby("job")["y"].apply(lambda s: (s ==1).mean())`) *before* the dataset is split.
2. **Preprocessing Data Leakage**: The `preprocessor.fit_transform(X)` is executed globally across the entire dataset *before* `train_test_split` is called.
3. **Cross-Validation Validity**: Because target encoding and preprocessing are applied globally to `X_processed`, the 5-fold cross-validation evaluates performance on validation folds that inherently contain scaling and target means heavily influenced by data outside those training folds.

## Why Each Issue is Problematic
- **Target Data Leakage**: By calculating the target mean for jobs using `y` across the whole dataset, the model receives information about the test set's target labels during the training phase. This artificially inflates performance and creates a falsely optimistic PR-AUC/ROC-AUC.
- **Preprocessing Data Leakage**: `StandardScaler` computes the feature `mean()` and `std()` to standardise data. Overlapping this onto the test set prior to the split allows the training process to "learn" the distribution metrics of the test set. In reality, future unseen data distributions are entirely unknown.
- **Cross-Validation Validity**: During 5-fold CV, the model should ideally simulate unseen data dynamically. Because features were preprocessed globally, every CV fold is highly contaminated, rendering the CV mean scores virtually useless for identifying generalisation capability.

## Expected Impact on Results
1. **Drop in Testing Metrics**: Expect both the Test PR-AUC and ROC-AUC test sets to systematically drop by a noticeable margin. Stripping the model of supernatural knowledge of test distributions removes the artificially inflated "over-confidence" it was riding on.
2. **CV Variation Matches Final Result**: With the `Pipeline` managing the transformations dynamically inside the CV loop bounds, the variance in your CV scores will closely follow your Hold-Out Test Split.

## Corrected Evaluation Procedure
To fix this, we strictly enforce that **data splitting happens first**. Feature engineering calculations and preprocessing `.fit()` must be executed **exclusively on the training subset**, and then mapped stringently to the test subset without recalculating metrics. Wrapping the preprocessor and estimator inside a `Pipeline` further ensures cross-validation handles standardisation on a true per-fold basis.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score, confusion_matrix

# 1. Load Data & Drop Artefacts
df = pd.read_csv('bank-additional-full.csv', sep=';')
df['contacted_previously'] = (df['pdays'] != 999).astype(int)
df = df.drop(columns=['pdays'])
df['y'] = df['y'].map({'no': 0, 'yes': 1})

# Define X and y FIRST
X = df.drop(columns=['y'])
y = df['y']

# 2. Strict Train/Test Split BEFORE any calculations
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3. Prevent Data Leakage during Target Encoding
X_train = X_train.copy()
X_test = X_test.copy()

# Calculate target encoding ONLY on training data
job_yes_rate = y_train.groupby(X_train["job"]).mean()

X_train["job_yes_rate"] = X_train["job"].map(job_yes_rate)
# Apply learned train rates to the test set. 
# Use train global mean for any unknown/missing test jobs.
X_test["job_yes_rate"] = X_test["job"].map(job_yes_rate).fillna(y_train.mean())

# Identify feature types
categorical_features = X_train.select_dtypes(include=['object']).columns.tolist()
numerical_features = X_train.select_dtypes(exclude=['object', 'datetime']).columns.tolist()
if 'contacted_previously' in numerical_features: numerical_features.remove('contacted_previously')
numerical_features.extend(['contacted_previously', 'job_yes_rate'])

# 4. Pipeline Construction 
# (Fit handles scaler parameter caching naturally per fold / per train set)
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', drop='if_binary'), categorical_features)
    ])

baseline_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'))
])

# 5. Robust Cross-Validation 
print("Running 5-Fold Stratified Cross-Validation on Train Data (Pipeline Safe)...")
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = cross_validate(
    baseline_pipeline, X_train, y_train, cv=cv, 
    scoring=['roc_auc', 'average_precision'],
    n_jobs=-1
)
print(f"CV ROC-AUC Mean: {cv_results['test_roc_auc'].mean():.4f}")
print(f"CV PR-AUC Mean: {cv_results['test_average_precision'].mean():.4f}")

# 6. Evaluation on Unseen Test Dataset
baseline_pipeline.fit(X_train, y_train)

y_pred = baseline_pipeline.predict(X_test)
y_pred_proba = baseline_pipeline.predict_proba(X_test)[:, 1]

print("\n--- HOLD-OUT TEST SET EVALUATION ---")
print("Classification Report:\n", classification_report(y_test, y_pred))
print(f"Test ROC-AUC: {roc_auc_score(y_test, y_pred_proba):.4f}")
print(f"Test PR-AUC: {average_precision_score(y_test, y_pred_proba):.4f}")
print(f"\nConfusion Matrix:\n{confusion_matrix(y_test, y_pred)}")

Running 5-Fold Stratified Cross-Validation on Train Data (Pipeline Safe)...


CV ROC-AUC Mean: 0.9355
CV PR-AUC Mean: 0.5809



--- HOLD-OUT TEST SET EVALUATION ---
Classification Report:
               precision    recall  f1-score   support

           0       0.99      0.86      0.92      7310
           1       0.45      0.91      0.60       928

    accuracy                           0.87      8238
   macro avg       0.72      0.89      0.76      8238
weighted avg       0.93      0.87      0.88      8238

Test ROC-AUC: 0.9438
Test PR-AUC: 0.6222

Confusion Matrix:
[[6283 1027]
 [  82  846]]
